In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("./data/clean_dataset.csv")
df

,locality,type,subtype,price,living_area,land_area,facades,state,furnished,terrace,garden,pool
0,Ladeuze,house,residence,90000,120,235,2,2,False,True,True,False
1,Charleroi,apartment,apartment,90000,49,0,2,6,False,True,False,False
2,Anvaing,house,residence,90000,165,1410,4,2,False,False,True,False
3,Dour,house,residence,90000,115,269,2,2,False,True,False,False
4,Élouges,house,residence,90000,113,170,2,2,False,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...
14608,Sint-Pieters-Woluwe,house,villa,2250000,365,1571,4,4,False,True,True,False
14609,Sint-Pieters-Woluwe,house,villa,2250000,375,1571,4,7,False,True,True,False
14610,Ukkel,apartment,penthouse,2250000,330,0,4,6,False,True,True,True
14611,Sint-Pieters-Woluwe,house,villa,2250000,365,1571,4,7,False,True,True,False


In [3]:

df.duplicated().sum()
dataset_dup = df[df.duplicated(keep=False)]
dataset_dup


,locality,type,subtype,price,living_area,land_area,facades,state,furnished,terrace,garden,pool
65,Bressoux,apartment,apartment,99000,63,0,2,4,False,False,False,False
97,Bressoux,apartment,apartment,99000,63,0,2,4,False,False,False,False
131,Charleroi,apartment,apartment,99999,100,0,4,4,False,True,False,False
132,Charleroi,apartment,apartment,99999,100,0,4,4,False,True,False,False
400,Charleroi,apartment,apartment,124900,90,0,2,4,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
14556,Kraainem,house,residence,1850000,513,1735,4,4,False,True,True,False
14559,Brussels,apartment,apartment,1859000,207,0,2,8,False,True,True,False
14560,Brussels,apartment,apartment,1859000,207,0,2,8,False,True,True,False
14563,Elsene,house,residence,1890000,417,230,2,7,False,True,True,False


In [4]:
df.isnull().sum()

locality       0
type           0
subtype        0
price          0
living_area    0
land_area      0
facades        0
state          0
furnished      0
terrace        0
garden         0
pool           0
dtype: int64

In [5]:
for col in ['locality', 'type', 'subtype']:
    if df[col].dtype == 'object':
        has_whitespace = df[col].str.contains('^\s|\s$', na=False).any()
        print(f"Whitespace in {col}: {has_whitespace}")

Whitespace in locality: False
Whitespace in type: False
Whitespace in subtype: False


In [7]:
municipality_to_region = {
    # ---- Brussels (19) ----
    'Anderlecht': 'Brussels',
    'Auderghem': 'Brussels',
    'Berchem-Sainte-Agathe': 'Brussels',
    'Bruxelles': 'Brussels',
    'Etterbeek': 'Brussels',
    'Evere': 'Brussels',
    'Forest': 'Brussels',
    'Ganshoren': 'Brussels',
    'Ixelles': 'Brussels',
    'Jette': 'Brussels',
    'Koekelberg': 'Brussels',
    'Molenbeek-Saint-Jean': 'Brussels',
    'Saint-Gilles': 'Brussels',
    'Saint-Josse-ten-Noode': 'Brussels',
    'Schaerbeek': 'Brussels',
    'Uccle': 'Brussels',
    'Watermael-Boitsfort': 'Brussels',
    'Woluwe-Saint-Lambert': 'Brussels',
    'Woluwe-Saint-Pierre': 'Brussels',

    # ---- Flanders (main list) ----
    'Aalst': 'Flanders',
    'Aalter': 'Flanders',
    'Aarschot': 'Flanders',
    'Aartselaar': 'Flanders',
    'Affligem': 'Flanders',
    'Alken': 'Flanders',
    'Alveringem': 'Flanders',
    'Antwerpen': 'Flanders',
    'Anzegem': 'Flanders',
    'Ardooie': 'Flanders',
    'Arendonk': 'Flanders',
    'As': 'Flanders',
    'Asse': 'Flanders',
    'Assenede': 'Flanders',
    'Avelgem': 'Flanders',
    'Baarle-Hertog': 'Flanders',
    'Balen': 'Flanders',
    'Beernem': 'Flanders',
    'Beerse': 'Flanders',
    'Beersel': 'Flanders',
    'Begijnendijk': 'Flanders',
    'Bekkevoort': 'Flanders',
    'Beringen': 'Flanders',
    'Berlaar': 'Flanders',
    'Berlare': 'Flanders',
    'Bertem': 'Flanders',
    'Bever': 'Flanders',
    'Beveren': 'Flanders',
    'Bierbeek': 'Flanders',
    'Bilzen': 'Flanders',
    'Blankenberge': 'Flanders',
    'Bocholt': 'Flanders',
    'Boechout': 'Flanders',
    'Bonheiden': 'Flanders',
    'Boom': 'Flanders',
    'Boortmeerbeek': 'Flanders',
    'Borgloon': 'Flanders',
    'Bornem': 'Flanders',
    'Borsbeek': 'Flanders',
    'Boutersem': 'Flanders',
    'Brakel': 'Flanders',
    'Brasschaat': 'Flanders',
    'Brecht': 'Flanders',
    'Bredene': 'Flanders',
    'Bree': 'Flanders',
    'Brugge': 'Flanders',
    'Buggenhout': 'Flanders',
    'Damme': 'Flanders',
    'De Haan': 'Flanders',
    'De Panne': 'Flanders',
    'De Pinte': 'Flanders',
    'Deerlijk': 'Flanders',
    'Deinze': 'Flanders',
    'Denderleeuw': 'Flanders',
    'Dendermonde': 'Flanders',
    'Dentergem': 'Flanders',
    'Dessel': 'Flanders',
    'Destelbergen': 'Flanders',
    'Diegem': 'Flanders',   # part of Machelen
    'Diepenbeek': 'Flanders',
    'Diest': 'Flanders',
    'Diksmuide': 'Flanders',
    'Dilbeek': 'Flanders',
    'Dilsen-Stokkem': 'Flanders',
    'Drogenbos': 'Flanders',
    'Duffel': 'Flanders',
    'Edegem': 'Flanders',
    'Eeklo': 'Flanders',
    'Erpe-Mere': 'Flanders',
    'Essen': 'Flanders',
    'Evergem': 'Flanders',
    'Galmaarden': 'Flanders',
    'Gavere': 'Flanders',
    'Geel': 'Flanders',
    'Geetbets': 'Flanders',
    'Genk': 'Flanders',
    'Gent': 'Flanders',
    'Geraardsbergen': 'Flanders',
    'Gingelom': 'Flanders',
    'Gistel': 'Flanders',
    'Glabbeek': 'Flanders',
    'Gooik': 'Flanders',
    'Grimbergen': 'Flanders',
    'Grobbendonk': 'Flanders',
    'Haacht': 'Flanders',
    'Haaltert': 'Flanders',
    'Halen': 'Flanders',
    'Halle': 'Flanders',
    'Ham': 'Flanders',
    'Hamme': 'Flanders',
    'Hamont-Achel': 'Flanders',
    'Harelbeke': 'Flanders',
    'Hasselt': 'Flanders',
    'Hechtel-Eksel': 'Flanders',
    'Heers': 'Flanders',
    'Heist-op-den-Berg': 'Flanders',
    'Hemiksem': 'Flanders',
    'Herent': 'Flanders',
    'Herentals': 'Flanders',
    'Herenthout': 'Flanders',
    'Herk-de-Stad': 'Flanders',
    'Herne': 'Flanders',
    'Herselt': 'Flanders',
    'Herstappe': 'Flanders',
    'Herve': 'Flanders',
    'Heusden-Zolder': 'Flanders',
    'Heuvelland': 'Flanders',
    'Hoegaarden': 'Flanders',
    'Hoeilaart': 'Flanders',
    'Hoeselt': 'Flanders',
    'Holsbeek': 'Flanders',
    'Hooglede': 'Flanders',
    'Hoogstraten': 'Flanders',
    'Horebeke': 'Flanders',
    'Houthalen-Helchteren': 'Flanders',
    'Houthulst': 'Flanders',
    'Hove': 'Flanders',
    'Huldenberg': 'Flanders',
    'Hulshout': 'Flanders',
    'Ichtegem': 'Flanders',
    'Ieper': 'Flanders',
    'Ingelmunster': 'Flanders',
    'Izegem': 'Flanders',
    'Jabbeke': 'Flanders',
    'Kalmthout': 'Flanders',
    'Kampenhout': 'Flanders',
    'Kapellen': 'Flanders',
    'Kapelle-op-den-Bos': 'Flanders',
    'Kasterlee': 'Flanders',
    'Keerbergen': 'Flanders',
    'Kinrooi': 'Flanders',
    'Kluisbergen': 'Flanders',
    'Knesselare': 'Flanders',
    'Knokke-Heist': 'Flanders',
    'Koekelare': 'Flanders',
    'Koksijde': 'Flanders',
    'Kontich': 'Flanders',
    'Kortemark': 'Flanders',
    'Kortenaken': 'Flanders',
    'Kortenberg': 'Flanders',
    'Kortessem': 'Flanders',
    'Kortrijk': 'Flanders',
    'Kraainem': 'Flanders',
    'Kruibeke': 'Flanders',
    'Kruishoutem': 'Flanders',
    'Kuurne': 'Flanders',
    'Laakdal': 'Flanders',
    'Laarne': 'Flanders',
    'Lanaken': 'Flanders',
    'Landen': 'Flanders',
    'Langemark-Poelkapelle': 'Flanders',
    'Lebbeke': 'Flanders',
    'Lede': 'Flanders',
    'Ledegem': 'Flanders',
    'Lendelede': 'Flanders',
    'Lennik': 'Flanders',
    'Leopoldsburg': 'Flanders',
    'Lepelstraat': 'Flanders',   # ?
    'Lessines': 'Wallonia',      # actually Lessen is Flemish name, but Lessines is Walloon? Wait Lessines is in Hainaut, Wallonia. So keep as Wallonia.
    'Leuven': 'Flanders',
    'Lichtervelde': 'Flanders',
    'Liedekerke': 'Flanders',
    'Lier': 'Flanders',
    'Lierde': 'Flanders',
    'Lievegem': 'Flanders',
    'Lille': 'Flanders',
    'Lint': 'Flanders',
    'Linter': 'Flanders',
    'Lochristi': 'Flanders',
    'Lokeren': 'Flanders',
    'Lommel': 'Flanders',
    'Londerzeel': 'Flanders',
    'Lo-Reninge': 'Flanders',
    'Lubbeek': 'Flanders',
    'Lummen': 'Flanders',
    'Maarkedal': 'Flanders',
    'Maaseik': 'Flanders',
    'Maasmechelen': 'Flanders',
    'Machelen': 'Flanders',
    'Maldegem': 'Flanders',
    'Malle': 'Flanders',
    'Mechelen': 'Flanders',
    'Meerhout': 'Flanders',
    'Meise': 'Flanders',
    'Melle': 'Flanders',
    'Menen': 'Flanders',
    'Merchtem': 'Flanders',
    'Merelbeke': 'Flanders',
    'Merksplas': 'Flanders',
    'Mesen': 'Flanders',
    'Meulebeke': 'Flanders',
    'Middelkerke': 'Flanders',
    'Moerbeke': 'Flanders',
    'Mol': 'Flanders',
    'Moorslede': 'Flanders',
    'Mortsel': 'Flanders',
    'Nazareth': 'Flanders',
    'Neerpelt': 'Flanders',   # part of Pelt now
    'Nevele': 'Flanders',
    'Niel': 'Flanders',
    'Nieuwerkerken': 'Flanders',
    'Nieuwpoort': 'Flanders',
    'Nijlen': 'Flanders',
    'Ninove': 'Flanders',
    'Olen': 'Flanders',
    'Oosterzele': 'Flanders',
    'Oostende': 'Flanders',
    'Oosterzele': 'Flanders',
    'Oostkamp': 'Flanders',
    'Oostrozebeke': 'Flanders',
    'Opwijk': 'Flanders',
    'Oud-Heverlee': 'Flanders',
    'Oudenaarde': 'Flanders',
    'Oudenburg': 'Flanders',
    'Oud-Turnhout': 'Flanders',
    'Overijse': 'Flanders',
    'Pecq': 'Flanders',      # Pecq is in Wallonia? Actually Pecq is in Hainaut, Wallonia. Wait, there is a Pecq in Wallonia and a similar name in Flanders? Let's check: Pecq is Walloon. I'll adjust later.
    'Pelt': 'Flanders',
    'Pepingen': 'Flanders',
    'Pittem': 'Flanders',
    'Poperinge': 'Flanders',
    'Putte': 'Flanders',
    'Puurs-Sint-Amands': 'Flanders',
    'Ranst': 'Flanders',
    'Ravels': 'Flanders',
    'Retie': 'Flanders',
    'Riemst': 'Flanders',
    'Rijkevorsel': 'Flanders',
    'Roeselare': 'Flanders',
    'Ronse': 'Flanders',
    'Roosdaal': 'Flanders',
    'Rotselaar': 'Flanders',
    'Ruiselede': 'Flanders',
    'Rumst': 'Flanders',
    'Schelle': 'Flanders',
    'Scherpenheuvel-Zichem': 'Flanders',
    'Schilde': 'Flanders',
    'Schoten': 'Flanders',
    'Sint-Amands': 'Flanders',
    'Sint-Genesius-Rode': 'Flanders',
    'Sint-Gillis-Waas': 'Flanders',
    'Sint-Katelijne-Waver': 'Flanders',
    'Sint-Laureins': 'Flanders',
    'Sint-Lievens-Houtem': 'Flanders',
    'Sint-Martens-Latem': 'Flanders',
    'Sint-Niklaas': 'Flanders',
    'Sint-Pieters-Leeuw': 'Flanders',
    'Sint-Truiden': 'Flanders',
    'Spiere-Helkijn': 'Flanders',
    'Stabroek': 'Flanders',
    'Staden': 'Flanders',
    'Steenokkerzeel': 'Flanders',
    'Stekene': 'Flanders',
    'Temse': 'Flanders',
    'Ternat': 'Flanders',
    'Tervuren': 'Flanders',
    'Tessenderlo': 'Flanders',
    'Tielt': 'Flanders',
    'Tielt-Winge': 'Flanders',
    'Tienen': 'Flanders',
    'Tongeren': 'Flanders',
    'Torhout': 'Flanders',
    'Tremelo': 'Flanders',
    'Turnhout': 'Flanders',
    'Veurne': 'Flanders',
    'Vilvoorde': 'Flanders',
    'Vleteren': 'Flanders',
    'Voeren': 'Flanders',
    'Vorselaar': 'Flanders',
    'Vosselaar': 'Flanders',
    'Waarschoot': 'Flanders',
    'Waasmunster': 'Flanders',
    'Wachtebeke': 'Flanders',
    'Waregem': 'Flanders',
    'Wellen': 'Flanders',
    'Wemmel': 'Flanders',
    'Wervik': 'Flanders',
    'Westerlo': 'Flanders',
    'Wetteren': 'Flanders',
    'Wevelgem': 'Flanders',
    'Wezembeek-Oppem': 'Flanders',
    'Wichelen': 'Flanders',
    'Wielsbeke': 'Flanders',
    'Wijnegem': 'Flanders',
    'Willebroek': 'Flanders',
    'Wingene': 'Flanders',
    'Wommelgem': 'Flanders',
    'Wortegem-Petegem': 'Flanders',
    'Wuustwezel': 'Flanders',
    'Zandhoven': 'Flanders',
    'Zaventem': 'Flanders',
    'Zedelgem': 'Flanders',
    'Zele': 'Flanders',
    'Zelzate': 'Flanders',
    'Zemst': 'Flanders',
    'Zoersel': 'Flanders',
    'Zomergem': 'Flanders',
    'Zonhoven': 'Flanders',
    'Zonnebeke': 'Flanders',
    'Zottegem': 'Flanders',
    'Zoutleeuw': 'Flanders',
    'Zuienkerke': 'Flanders',
    'Zulte': 'Flanders',
    'Zutendaal': 'Flanders',
    'Zwalm': 'Flanders',
    'Zwevegem': 'Flanders',
    'Zwijndrecht': 'Flanders',

    # ---- Wallonia (including German-speaking) ----
    'Aiseau-Presles': 'Wallonia',
    'Amay': 'Wallonia',
    'Amel': 'Wallonia',
    'Andenne': 'Wallonia',
    'Anderlues': 'Wallonia',
    'Anhée': 'Wallonia',
    'Ans': 'Wallonia',
    'Anthisnes': 'Wallonia',
    'Antoing': 'Wallonia',
    'Arlon': 'Wallonia',
    'Assesse': 'Wallonia',
    'Ath': 'Wallonia',
    'Attert': 'Wallonia',
    'Aubange': 'Wallonia',
    'Aubel': 'Wallonia',
    'Awans': 'Wallonia',
    'Aywaille': 'Wallonia',
    'Baelen': 'Wallonia',
    'Bassenge': 'Wallonia',
    'Bastogne': 'Wallonia',
    'Beaumont': 'Wallonia',
    'Beauraing': 'Wallonia',
    'Beauvechain': 'Wallonia',
    'Beloeil': 'Wallonia',
    'Berloz': 'Wallonia',
    'Bernissart': 'Wallonia',
    'Bertogne': 'Wallonia',
    'Bertrix': 'Wallonia',
    'Beyne-Heusay': 'Wallonia',
    'Bièvre': 'Wallonia',
    'Binche': 'Wallonia',
    'Blegny': 'Wallonia',
    'Bouillon': 'Wallonia',
    'Boussu': 'Wallonia',
    'Braine-le-Château': 'Wallonia',
    'Braine-le-Comte': 'Wallonia',
    'Braine-l\'Alleud': 'Wallonia',
    'Braives': 'Wallonia',
    'Brugelette': 'Wallonia',
    'Brunehaut': 'Wallonia',
    'Büllingen': 'Wallonia',
    'Burdinne': 'Wallonia',
    'Burg-Reuland': 'Wallonia',
    'Bütgenbach': 'Wallonia',
    'Celles': 'Wallonia',
    'Cerfontaine': 'Wallonia',
    'Chapelle-lez-Herlaimont': 'Wallonia',
    'Charleroi': 'Wallonia',
    'Chastre': 'Wallonia',
    'Châtelet': 'Wallonia',
    'Chaudfontaine': 'Wallonia',
    'Chaumont-Gistoux': 'Wallonia',
    'Chièvres': 'Wallonia',
    'Chimay': 'Wallonia',
    'Chiny': 'Wallonia',
    'Ciney': 'Wallonia',
    'Clavier': 'Wallonia',
    'Colfontaine': 'Wallonia',
    'Comblain-au-Pont': 'Wallonia',
    'Comines-Warneton': 'Wallonia',
    'Courcelles': 'Wallonia',
    'Court-Saint-Étienne': 'Wallonia',
    'Couvin': 'Wallonia',
    'Crisnée': 'Wallonia',
    'Dalhem': 'Wallonia',
    'Daverdisse': 'Wallonia',
    'Dinant': 'Wallonia',
    'Dison': 'Wallonia',
    'Doische': 'Wallonia',
    'Donceel': 'Wallonia',
    'Dour': 'Wallonia',
    'Durbuy': 'Wallonia',
    'Écaussinnes': 'Wallonia',
    'Éghezée': 'Wallonia',
    'Ellezelles': 'Wallonia',
    'Enghien': 'Wallonia',
    'Engis': 'Wallonia',
    'Érezée': 'Wallonia',
    'Erquelinnes': 'Wallonia',
    'Esneux': 'Wallonia',
    'Estaimpuis': 'Wallonia',
    'Estinnes': 'Wallonia',
    'Étalle': 'Wallonia',
    'Faimes': 'Wallonia',
    'Farciennes': 'Wallonia',
    'Fauvillers': 'Wallonia',
    'Fernelmont': 'Wallonia',
    'Ferrières': 'Wallonia',
    'Fexhe-le-Haut-Clocher': 'Wallonia',
    'Flémalle': 'Wallonia',
    'Fléron': 'Wallonia',
    'Flobecq': 'Wallonia',
    'Floreffe': 'Wallonia',
    'Florennes': 'Wallonia',
    'Florenville': 'Wallonia',
    'Fontaine-l\'Évêque': 'Wallonia',
    'Fosses-la-Ville': 'Wallonia',
    'Frameries': 'Wallonia',
    'Frasnes-lez-Anvaing': 'Wallonia',
    'Froidchapelle': 'Wallonia',
    'Gedinne': 'Wallonia',
    'Geer': 'Wallonia',
    'Gembloux': 'Wallonia',
    'Genappe': 'Wallonia',
    'Gerpinnes': 'Wallonia',
    'Gesves': 'Wallonia',
    'Gouvy': 'Wallonia',
    'Grâce-Hollogne': 'Wallonia',
    'Grez-Doiceau': 'Wallonia',
    'Habay': 'Wallonia',
    'Hamoir': 'Wallonia',
    'Ham-sur-Heure-Nalinnes': 'Wallonia',
    'Hannut': 'Wallonia',
    'Hastière': 'Wallonia',
    'Havelange': 'Wallonia',
    'Hélécine': 'Wallonia',
    'Hensies': 'Wallonia',
    'Herbeumont': 'Wallonia',
    'Herstal': 'Wallonia',
    'Herve': 'Wallonia',
    'Honnelles': 'Wallonia',
    'Hotton': 'Wallonia',
    'Houffalize': 'Wallonia',
    'Houyet': 'Wallonia',
    'Huy': 'Wallonia',
    'Incourt': 'Wallonia',
    'Ittre': 'Wallonia',
    'Jalhay': 'Wallonia',
    'Jemeppe-sur-Sambre': 'Wallonia',
    'Jodoigne': 'Wallonia',
    'Juprelle': 'Wallonia',
    'Jurbise': 'Wallonia',
    'La Bruyère': 'Wallonia',
    'La Louvière': 'Wallonia',
    'La Roche-en-Ardenne': 'Wallonia',
    'Lanaken': 'Wallonia',   # actually Lanaken is Flemish? Wait Lanaken is in Limburg, Flanders. So it should be Flanders. I need to be careful. I'll move it.
    'Lasne': 'Wallonia',
    'Le Rœulx': 'Wallonia',
    'Léglise': 'Wallonia',
    'Les Bons Villers': 'Wallonia',
    'Lessines': 'Wallonia',
    'Leuze-en-Hainaut': 'Wallonia',
    'Libin': 'Wallonia',
    'Libramont-Chevigny': 'Wallonia',
    'Lierneux': 'Wallonia',
    'Limbourg': 'Wallonia',
    'Lincent': 'Wallonia',
    'Lobbes': 'Wallonia',
    'Lontzen': 'Wallonia',
    'Louvain-la-Neuve': 'Wallonia',
    'Malmedy': 'Wallonia',
    'Manage': 'Wallonia',
    'Manhay': 'Wallonia',
    'Marche-en-Famenne': 'Wallonia',
    'Marchin': 'Wallonia',
    'Martelange': 'Wallonia',
    'Meix-devant-Virton': 'Wallonia',
    'Mettet': 'Wallonia',
    'Modave': 'Wallonia',
    'Momignies': 'Wallonia',
    'Mons': 'Wallonia',
    'Mont-de-l\'Enclus': 'Wallonia',
    'Montigny-le-Tilleul': 'Wallonia',
    'Mont-Saint-Guibert': 'Wallonia',
    'Morlanwelz': 'Wallonia',
    'Mouscron': 'Wallonia',
    'Musson': 'Wallonia',
    'Namur': 'Wallonia',
    'Nandrin': 'Wallonia',
    'Nassogne': 'Wallonia',
    'Neufchâteau': 'Wallonia',
    'Neupré': 'Wallonia',
    'Nivelles': 'Wallonia',
    'Ohey': 'Wallonia',
    'Onhaye': 'Wallonia',
    'Oreye': 'Wallonia',
    'Orp-Jauche': 'Wallonia',
    'Ottignies-Louvain-la-Neuve': 'Wallonia',
    'Ouffet': 'Wallonia',
    'Oupeye': 'Wallonia',
    'Paliseul': 'Wallonia',
    'Pecq': 'Wallonia',
    'Pepinster': 'Wallonia',
    'Péruwelz': 'Wallonia',
    'Perwez': 'Wallonia',
    'Philippeville': 'Wallonia',
    'Plombières': 'Wallonia',
    'Pont-à-Celles': 'Wallonia',
    'Profondeville': 'Wallonia',
    'Quaregnon': 'Wallonia',
    'Quévy': 'Wallonia',
    'Quiévrain': 'Wallonia',
    'Raeren': 'Wallonia',
    'Ramillies': 'Wallonia',
    'Rebecq': 'Wallonia',
    'Remicourt': 'Wallonia',
    'Rendeux': 'Wallonia',
    'Rixensart': 'Wallonia',
    'Rochefort': 'Wallonia',
    'Rouvroy': 'Wallonia',
    'Saint-Georges-sur-Meuse': 'Wallonia',
    'Saint-Ghislain': 'Wallonia',
    'Saint-Hubert': 'Wallonia',
    'Saint-Léger': 'Wallonia',
    'Saint-Nicolas': 'Wallonia',
    'Saint-Vith': 'Wallonia',
    'Sambreville': 'Wallonia',
    'Sankt Vith': 'Wallonia',
    'Seneffe': 'Wallonia',
    'Seraing': 'Wallonia',
    'Silly': 'Wallonia',
    'Sivry-Rance': 'Wallonia',
    'Soignies': 'Wallonia',
    'Sombreffe': 'Wallonia',
    'Somme-Leuze': 'Wallonia',
    'Soumagne': 'Wallonia',
    'Spa': 'Wallonia',
    'Sprimont': 'Wallonia',
    'Stavelot': 'Wallonia',
    'Stoumont': 'Wallonia',
    'Tellin': 'Wallonia',
    'Tenneville': 'Wallonia',
    'Theux': 'Wallonia',
    'Thimister-Clermont': 'Wallonia',
    'Thuin': 'Wallonia',
    'Tinlot': 'Wallonia',
    'Tintigny': 'Wallonia',
    'Tournai': 'Wallonia',
    'Trois-Ponts': 'Wallonia',
    'Trooz': 'Wallonia',
    'Tubize': 'Wallonia',
    'Vaux-sur-Sûre': 'Wallonia',
    'Verlaine': 'Wallonia',
    'Verviers': 'Wallonia',
    'Vielsalm': 'Wallonia',
    'Villers-la-Ville': 'Wallonia',
    'Villers-le-Bouillet': 'Wallonia',
    'Viroinval': 'Wallonia',
    'Virton': 'Wallonia',
    'Visé': 'Wallonia',
    'Vresse-sur-Semois': 'Wallonia',
    'Waimes': 'Wallonia',
    'Walcourt': 'Wallonia',
    'Walhain': 'Wallonia',
    'Wanze': 'Wallonia',
    'Waremme': 'Wallonia',
    'Wasseiges': 'Wallonia',
    'Waterloo': 'Wallonia',
    'Wavre': 'Wallonia',
    'Welkenraedt': 'Wallonia',
    'Wellin': 'Wallonia',
    'Yvoir': 'Wallonia',
        'Aalbeke': 'Flanders',
    'Aarsele': 'Flanders',
    'Aartrijke': 'Flanders',
    'Achel': 'Flanders',
    'Achêne': 'Wallonia',
    'Acoz': 'Wallonia',
    'Adegem': 'Flanders',
    'Afsnee': 'Flanders',
    'Agimont': 'Wallonia',
    'Aische-en-Refail': 'Wallonia',
    'Aiseau': 'Wallonia',
    'Aisemont': 'Wallonia',
    'Alle': 'Wallonia',
    'Alleur': 'Wallonia',
    'Alsemberg': 'Flanders',
    'Ambly': 'Wallonia',
    'Andrimont': 'Wallonia',
    'Angleur': 'Wallonia',
    'Angre': 'Wallonia',
    'Angreau': 'Wallonia',
    'Anseremme': 'Wallonia',
    'Anseroeul': 'Wallonia',
    'Antheit': 'Wallonia',
    'Antwerp': 'Flanders',
    'Anvaing': 'Wallonia',
    'Appelterre-Eichem': 'Flanders',
    'Appels': 'Flanders',
    'Archennes': 'Wallonia',
    'Argenteau': 'Wallonia',
    'Arquennes': 'Wallonia',
    'Arsimont': 'Wallonia',
    'Asper': 'Flanders',
    'Assebroek': 'Flanders',
    'Astene': 'Flanders',
    'Athus': 'Wallonia',
    'Attre': 'Wallonia',
    'Auby-sur-Semois': 'Wallonia',
    'Aulnois': 'Wallonia',
    'Autre-Église': 'Wallonia',
    'Auvelais': 'Wallonia',
    'Averbode': 'Flanders',
    'Avin': 'Wallonia',
    'Awenne': 'Wallonia',
    'Awirs': 'Wallonia',
    'Aye': 'Wallonia',
    'Ayeneux': 'Wallonia',

    # B
    'Baal': 'Flanders',
    'Baardegem': 'Flanders',
    'Bailleul': 'Wallonia',
    'Baillonville': 'Wallonia',
    'Baisieux': 'Wallonia',
    'Baisy-Thy': 'Wallonia',
    'Balâtre': 'Wallonia',
    'Bande': 'Wallonia',
    'Barbençon': 'Wallonia',
    'Barchon': 'Wallonia',
    'Baronville': 'Wallonia',
    'Barvaux-sur-Ourthe': 'Wallonia',
    'Bas-Oha': 'Wallonia',
    'Bas-Warneton': 'Wallonia',
    'Basècles': 'Wallonia',
    'Bassilly': 'Wallonia',
    'Battice': 'Wallonia',
    'Baudour': 'Wallonia',
    'Baulers': 'Wallonia',
    'Bavegem': 'Flanders',
    'Bavikhove': 'Flanders',
    'Bazel': 'Flanders',
    'Beaufays': 'Wallonia',
    'Beauwelz': 'Wallonia',
    'Beausaint': 'Wallonia',
    'Beclers': 'Wallonia',
    'Beez': 'Wallonia',
    'Beffe': 'Wallonia',
    'Beerlegem': 'Flanders',
    'Beervelde': 'Flanders',
    'Beerzel': 'Flanders',
    'Beert': 'Flanders',
    'Beigem': 'Flanders',
    'Bekegem': 'Flanders',
    'Belgrade': 'Wallonia',
    'Bellecourt': 'Wallonia',
    'Bellefontaine': 'Wallonia',
    'Bellem': 'Flanders',
    'Bellegem': 'Flanders',
    'Bellevaux-Ligneuville': 'Wallonia',
    'Belsele': 'Flanders',
    'Ben-Ahin': 'Wallonia',
    'Bende': 'Wallonia',
    'Berchem': 'Flanders',
    'Berg': 'Flanders',
    'Bergilers': 'Wallonia',
    'Berneau': 'Wallonia',
    'Bersillies-l\'Abbaye': 'Wallonia',
    'Berzée': 'Wallonia',
    'Bettincourt': 'Wallonia',
    'Beuzet': 'Wallonia',
    'Bevel': 'Flanders',
    'Bevere': 'Flanders',
    'Beveren-Aan-Den-Ijzer': 'Flanders',
    'Beveren-Leie': 'Flanders',
    'Beverlo': 'Flanders',
    'Beverst': 'Flanders',
    'Bierges': 'Wallonia',
    'Bierghes': 'Wallonia',
    'Bierwart': 'Wallonia',
    'Biesme': 'Wallonia',
    'Biesme-sous-Thuin': 'Wallonia',
    'Bizet': 'Wallonia',
    'Blaugies': 'Wallonia',
    'Blandain': 'Wallonia',
    'Blaton': 'Wallonia',
    'Bleid': 'Wallonia',
    'Blégny': 'Wallonia',
    'Bléharies': 'Wallonia',
    'Boezinge': 'Flanders',
    'Bohan': 'Wallonia',
    'Boignée': 'Wallonia',
    'Boirs': 'Wallonia',
    'Bois-d\'Haine': 'Wallonia',
    'Bois-de-Villers': 'Wallonia',
    'Bois-et-Borsu': 'Wallonia',
    'Bolland': 'Wallonia',
    'Bon-Secours': 'Wallonia',
    'Boncelles': 'Wallonia',
    'Bonlez': 'Wallonia',
    'Bonneville': 'Wallonia',
    'Bonsin': 'Wallonia',
    'Booischot': 'Flanders',
    'Borgerhout': 'Flanders',
    'Borsbeke': 'Flanders',
    'Bossut-Gottechain': 'Wallonia',
    'Bost': 'Flanders',
    'Bottelare': 'Flanders',
    'Bouffioulx': 'Wallonia',
    'Bouge': 'Wallonia',
    'Bourlers': 'Wallonia',
    'Boussoit': 'Wallonia',
    'Boussu-en-Fagne': 'Wallonia',
    'Boussu-lez-Walcourt': 'Wallonia',
    'Bousval': 'Wallonia',
    'Bouvignies': 'Wallonia',
    'Bovesse': 'Wallonia',
    'Brasmenil': 'Wallonia',
    'Bray': 'Wallonia',
    'Breendonk': 'Flanders',
    'Bressoux': 'Wallonia',
    'Brielen': 'Flanders',
    'Broechem': 'Flanders',
    'Brussegem': 'Flanders',
    'Brussels': 'Brussels-Capital',
    'Bruyelle': 'Wallonia',
    'Budingen': 'Flanders',
    'Buissenal': 'Wallonia',
    'Buizingen': 'Flanders',
    'Burcht': 'Flanders',
    'Bure': 'Wallonia',
    'Buzet': 'Wallonia',

    # C
    'Cambron-Saint-Vincent': 'Wallonia',
    'Carnières': 'Wallonia',
    'Casteau': 'Wallonia',
    'Céroux-Mousty': 'Wallonia',
    'Champion': 'Wallonia',
    'Chapelle-à-Wattines': 'Wallonia',
    'Charneux': 'Wallonia',
    'Chastre-Villeroux-Blanmont': 'Wallonia',
    'Chastrès': 'Wallonia',
    'Châtelineau': 'Wallonia',
    'Châtillon': 'Wallonia',
    'Chaussée-Notre-Dame-Louvignies': 'Wallonia',
    'Chênée': 'Wallonia',
    'Cheratte': 'Wallonia',
    'Chercq': 'Wallonia',
    'Chevron': 'Wallonia',
    'Ciergnon': 'Wallonia',
    'Ciply': 'Wallonia',
    'Clabecq': 'Wallonia',
    'Clermont': 'Wallonia',
    'Comblain-Fairon': 'Wallonia',
    'Comines': 'Wallonia',
    'Corbais': 'Wallonia',
    'Corbion': 'Wallonia',
    'Cornesse': 'Wallonia',
    'Corroy-le-Grand': 'Wallonia',
    'Cortil-Noirmont': 'Wallonia',
    'Couillet': 'Wallonia',
    'Cour-sur-Heure': 'Wallonia',
    'Courrière': 'Wallonia',
    'Couthuin': 'Wallonia',
    'Cras-Avernas': 'Wallonia',
    'Crombach': 'Wallonia',
    'Cuesmes': 'Wallonia',
    'Cul-des-Sarts': 'Wallonia',

    # D
    'Dadizele': 'Flanders',
    'Dailly': 'Wallonia',
    'Dampicourt': 'Wallonia',
    'Dampremy': 'Wallonia',
    'Daussois': 'Wallonia',
    'Dave': 'Wallonia',
    'De Klinge': 'Flanders',
    'Deftinge': 'Flanders',
    'Denderhoutem': 'Flanders',
    'Denderwindeke': 'Flanders',
    'Denée': 'Wallonia',
    'Dergneau': 'Wallonia',
    'Desselgem': 'Flanders',
    'Desteldonk': 'Flanders',
    'Deurne': 'Flanders',
    'Deux-Acren': 'Wallonia',
    'Dhuy': 'Wallonia',
    'Dikkebus': 'Flanders',
    'Dikkelvenne': 'Flanders',
    'Dilsen': 'Flanders',
    'Dion': 'Wallonia',
    'Dochamps': 'Wallonia',
    'Doel': 'Flanders',
    'Dohan': 'Wallonia',
    'Dolembreux': 'Wallonia',
    'Donstiennes': 'Wallonia',
    'Dottignies': 'Wallonia',
    'Drongen': 'Flanders',
    'Duinbergen': 'Flanders',
    'Duisburg': 'Flanders',
    'Dworp': 'Flanders',

    # E
    'Ecaussinnes': 'Wallonia',
    'Ecaussinnes-d\'Enghien': 'Wallonia',
    'Écaussinnes-Lalaing': 'Wallonia',
    'Eernegem': 'Flanders',
    'Eindhout': 'Flanders',
    'Ekeren': 'Flanders',
    'Eksel': 'Flanders',
    'Elewijt': 'Flanders',
    'Ellignies-Sainte-Anne': 'Wallonia',
    'Élouges': 'Wallonia',
    'Elsene': 'Brussels-Capital',
    'Elverdinge': 'Flanders',
    'Elversele': 'Flanders',
    'Emblem': 'Flanders',
    'Embourg': 'Wallonia',
    'Emelgem': 'Flanders',
    'Énines': 'Wallonia',
    'Ename': 'Flanders',
    'Ensival': 'Wallonia',
    'Épinois': 'Wallonia',
    'Eppegem': 'Flanders',
    'Éprave': 'Wallonia',
    'Ère': 'Wallonia',
    'Erembodegem': 'Flanders',
    'Ernage': 'Wallonia',
    'Erpe': 'Flanders',
    'Erpent': 'Wallonia',
    'Erps-Kwerps': 'Flanders',
    'Ertvelde': 'Flanders',
    'Erwetegem': 'Flanders',
    'Escanaffles': 'Wallonia',
    'Estaimbourg': 'Wallonia',
    'Estinnes-au-Mont': 'Wallonia',
    'Eugies': 'Wallonia',
    'Eupen': 'Wallonia',
    'Everberg': 'Flanders',

    # F
    'Falaën': 'Wallonia',
    'Falisolle': 'Wallonia',
    'Falmignoul': 'Wallonia',
    'Familleureux': 'Wallonia',
    'Faulx-les-Tombes': 'Wallonia',
    'Faymonville': 'Wallonia',
    'Fayt-lez-Manage': 'Wallonia',
    'Felenne': 'Wallonia',
    'Feluy': 'Wallonia',
    'Feneur': 'Wallonia',
    'Filot': 'Wallonia',
    'Fize-Fontaine': 'Wallonia',
    'Flawinne': 'Wallonia',
    'Flémalle-Grande': 'Wallonia',
    'Flénu': 'Wallonia',
    'Fleurus': 'Wallonia',
    'Florée': 'Wallonia',
    'Flostoy': 'Wallonia',
    'Folx-les-Caves': 'Wallonia',
    'Forchies-la-Marche': 'Wallonia',
    'Forêt': 'Wallonia',
    'Forges': 'Wallonia',
    'Forville': 'Wallonia',
    'Fraire': 'Wallonia',
    'Framont': 'Wallonia',
    'Francorchamps': 'Wallonia',
    'Franière': 'Wallonia',
    'Frasnes': 'Wallonia',
    'Frasnes-lez-Gosselies': 'Wallonia',
    'Froidmont': 'Wallonia',
    'Froyennes': 'Wallonia',

    # G
    'Gaurain-Ramecroix': 'Wallonia',
    'Genly': 'Wallonia',
    'Gentbrugge': 'Flanders',
    'Gentinnes': 'Wallonia',
    'Genval': 'Wallonia',
    'Gérouville': 'Wallonia',
    'Ghlin': 'Wallonia',
    'Ghislenghien': 'Wallonia',
    'Ghoy': 'Wallonia',
    'Gierle': 'Flanders',
    'Gijzegem': 'Flanders',
    'Gilly': 'Wallonia',
    'Gimnée': 'Wallonia',
    'Givry': 'Wallonia',
    'Glabais': 'Wallonia',
    'Glain': 'Wallonia',
    'Glons': 'Wallonia',
    'Godarville': 'Wallonia',
    'Godinne': 'Wallonia',
    'Goé': 'Wallonia',
    'Goeferdinge': 'Flanders',
    'Goesnes': 'Wallonia',
    'Gomzé-Andoumont': 'Wallonia',
    'Gooreind': 'Flanders',
    'Gosselies': 'Wallonia',
    'Goutroux': 'Wallonia',
    'Gouy-lez-Piéton': 'Wallonia',
    'Gozée': 'Wallonia',
    'Grâce-Berleur': 'Wallonia',
    'Grand-Hallet': 'Wallonia',
    'Grand-Halleux': 'Wallonia',
    'Grand-Leez': 'Wallonia',
    'Grand-Reng': 'Wallonia',
    'Grandrieu': 'Wallonia',
    'Grimminge': 'Flanders',
    'Grivegnée': 'Wallonia',
    'Groot-Bijgaarden': 'Flanders',
    'Grote-Brogel': 'Flanders',
    'Guigoven': 'Flanders',
    'Gullegem': 'Flanders',

    # H
    'Haasdonk': 'Flanders',
    'Haasrode': 'Flanders',
    'Habay-la-Neuve': 'Wallonia',
    'Haccourt': 'Wallonia',
    'Hacquegnies': 'Wallonia',
    'Haillot': 'Wallonia',
    'Haine-Saint-Paul': 'Wallonia',
    'Haine-Saint-Pierre': 'Wallonia',
    'Hainin': 'Wallonia',
    'Halanzy': 'Wallonia',
    'Hallaar': 'Flanders',
    'Halma': 'Wallonia',
    'Ham-sur-Heure': 'Wallonia',
    'Ham-sur-Sambre': 'Wallonia',
    'Hamipré': 'Wallonia',
    'Hamme-Mille': 'Wallonia',
    'Hamois': 'Wallonia',
    'Hamont': 'Flanders',
    'Handzame': 'Flanders',
    'Han-sur-Lesse': 'Wallonia',
    'Hansbeke': 'Flanders',
    'Hanzinelle': 'Wallonia',
    'Hanzinne': 'Wallonia',
    'Harchies': 'Wallonia',
    'Haren': 'Brussels-Capital',
    'Hargimont': 'Wallonia',
    'Harmignies': 'Wallonia',
    'Harzé': 'Wallonia',
    'Hastière-Lavaux': 'Wallonia',
    'Haut-Fays': 'Wallonia',
    'Haut-Ittre': 'Wallonia',
    'Hautrage': 'Wallonia',
    'Havay': 'Wallonia',
    'Havinnes': 'Wallonia',
    'Havré': 'Wallonia',
    'Hechtel': 'Flanders',
    'Heer': 'Wallonia',
    'Heffen': 'Flanders',
    'Heist': 'Flanders',
    'Helchteren': 'Flanders',
    'Helkijn': 'Flanders',
    'Hennuyères': 'Wallonia',
    'Heppen': 'Flanders',
    'Herfelingen': 'Flanders',
    'Hérinnes': 'Flanders',
    'Hermalle-sous-Argenteau': 'Wallonia',
    'Hermalle-sous-Huy': 'Wallonia',
    'Hermée': 'Wallonia',
    'Héron': 'Wallonia',
    'Herseaux': 'Wallonia',
    'Hertain': 'Wallonia',
    'Herzele': 'Flanders',
    'Heule': 'Flanders',
    'Heure-le-Romain': 'Wallonia',
    'Heusden': 'Flanders',
    'Heusy': 'Wallonia',
    'Hévillers': 'Wallonia',
    'Hever': 'Flanders',
    'Heverlee': 'Flanders',
    'Hillegem': 'Flanders',
    'Hingene': 'Flanders',
    'Hoboken': 'Flanders',
    'Hody': 'Wallonia',
    'Hoevenen': 'Flanders',
    'Hofstade': 'Flanders',
    'Hogne': 'Wallonia',
    'Hognoul': 'Wallonia',
    'Hollebeke': 'Flanders',
    'Hollain': 'Wallonia',
    'Hombeek': 'Flanders',
    'Honnay': 'Wallonia',
    'Hornu': 'Wallonia',
    'Houdemont': 'Wallonia',
    'Houdeng-Aimeries': 'Wallonia',
    'Houdeng-Goegnies': 'Wallonia',
    'Housse': 'Wallonia',
    'Houtain-le-Val': 'Wallonia',
    'Houtain-Saint-Siméon': 'Wallonia',
    'Houtaing': 'Wallonia',
    'Houthalen': 'Flanders',
    'Huissignies': 'Wallonia',
    'Huizingen': 'Flanders',
    'Hulste': 'Flanders',
    'Humbeek': 'Flanders',
    'Hyon': 'Wallonia',

    # I
    'Iddergem': 'Flanders',
    'Idegem': 'Flanders',
    'Irchonwelz': 'Wallonia',
    'Itegem': 'Flanders',
    'Itterbeek': 'Flanders',
    'Ivoz-Ramet': 'Wallonia',
    'Izel': 'Wallonia',
    'Izier': 'Wallonia',

    # J
    'Jamagne': 'Wallonia',
    'Jambes': 'Wallonia',
    'Jandrain-Jandrenouille': 'Wallonia',
    'Jauche': 'Wallonia',
    'Jemappes': 'Wallonia',
    'Jemelle': 'Wallonia',
    'Jemeppe': 'Wallonia',
    'Jemeppe-Sur-Sambre': 'Wallonia',
    'Jumet': 'Wallonia',
    'Jupille': 'Wallonia',

    # K
    'Kain': 'Wallonia',
    'Kalken': 'Flanders',
    'Kaulille': 'Flanders',
    'Kemzeke': 'Flanders',
    'Kerkhove': 'Flanders',
    'Kerksken': 'Flanders',
    'Kermt': 'Flanders',
    'Kessel': 'Flanders',
    'Kessel-Lo': 'Flanders',
    'Kieldrecht': 'Flanders',
    'Klerken': 'Flanders',
    'Knokke': 'Flanders',
    'Kobbegem': 'Flanders',
    'Koersel': 'Flanders',
    'Koningshooikt': 'Flanders',
    'Kortijs': 'Flanders',
    'Kuringen': 'Flanders',

    # L
    'La Bouverie': 'Wallonia',
    'La Glanerie': 'Wallonia',
    'La Gleize': 'Wallonia',
    'La Hestre': 'Wallonia',
    'La Hulpe': 'Wallonia',
    'La Reid': 'Wallonia',
    'Labuissière': 'Wallonia',
    'Lacuisine': 'Wallonia',
    'Ladeuze': 'Wallonia',
    'Laken': 'Brussels-Capital',
    'Lamain': 'Wallonia',
    'Lambermont': 'Wallonia',
    'Lambusart': 'Wallonia',
    'Lamine': 'Wallonia',
    'Lamorteau': 'Wallonia',
    'Lanaye': 'Wallonia',
    'Laneffe': 'Wallonia',
    'Langdorp': 'Flanders',
    'Langemark': 'Flanders',
    'Laplaigne': 'Wallonia',
    'Lathuy': 'Wallonia',
    'Latinne': 'Wallonia',
    'Lauwe': 'Flanders',
    'Lavaux-Sainte-Anne': 'Wallonia',
    'Lavoir': 'Wallonia',
    'Le Mesnil': 'Wallonia',
    'Le Roeulx': 'Wallonia',
    'Le Roux': 'Wallonia',
    'Ledeberg': 'Flanders',
    'Leefdaal': 'Flanders',
    'Leernes': 'Wallonia',
    'Leers-Nord': 'Wallonia',
    'Leers-et-Fosteau': 'Wallonia',
    'Leest': 'Flanders',
    'Lembeek': 'Flanders',
    'Lembeke': 'Flanders',
    'Lens': 'Wallonia',
    'Lens-Saint-Remy': 'Wallonia',
    'Les Bulles': 'Wallonia',
    'Les Hayons': 'Wallonia',
    'Lesdain': 'Wallonia',
    'Lesterny': 'Wallonia',
    'Leupegem': 'Flanders',
    'Leval-Chaudeville': 'Wallonia',
    'Leval-Trahegnies': 'Wallonia',
    'Lichtaart': 'Flanders',
    'Liers': 'Wallonia',
    'Ligne': 'Wallonia',
    'Ligny': 'Wallonia',
    'Lillois-Witterzée': 'Wallonia',
    'Limal': 'Wallonia',
    'Limelette': 'Wallonia',
    'Linden': 'Flanders',
    'Linkebeek': 'Flanders',
    'Lippelo': 'Flanders',
    'Lissewege': 'Flanders',
    'Lixhe': 'Wallonia',
    'Lodelinsart': 'Wallonia',
    'Lombardsijde': 'Flanders',
    'Lombise': 'Wallonia',
    'Longlier': 'Wallonia',
    'Longueville': 'Wallonia',
    'Loncin': 'Wallonia',
    'Lonzée': 'Wallonia',
    'Loonbeek': 'Flanders',
    'Loppem': 'Flanders',
    'Lot': 'Flanders',
    'Louveigné': 'Wallonia',
    'Lovendegem': 'Flanders',
    'Loverval': 'Wallonia',
    'Luingne': 'Wallonia',
    'Luttre': 'Wallonia',

    # M
    'Macon': 'Wallonia',
    'Macquenoise': 'Wallonia',
    'Maffle': 'Wallonia',
    'Maisières': 'Wallonia',
    'Maissin': 'Wallonia',
    'Malderen': 'Flanders',
    'Malonne': 'Wallonia',
    'Marbais': 'Wallonia',
    'Marchienne-au-Pont': 'Wallonia',
    'Marcinelle': 'Wallonia',
    'Marcourt': 'Wallonia',
    'Mariakerke': 'Flanders',
    'Mariembourg': 'Wallonia',
    'Marilles': 'Wallonia',
    'Marke': 'Flanders',
    'Marquain': 'Wallonia',
    'Masnuy-Saint-Pierre': 'Wallonia',
    'Massenhoven': 'Flanders',
    'Massemen': 'Flanders',
    'Matagne-la-Grande': 'Wallonia',
    'Maurage': 'Wallonia',
    'Mazenzele': 'Flanders',
    'Mazy': 'Wallonia',
    'Méan': 'Wallonia',
    'Mechelen-aan-de-Maas': 'Flanders',
    'Meeffe': 'Wallonia',
    'Meer': 'Flanders',
    'Meerbeke': 'Flanders',
    'Meerdonk': 'Flanders',
    'Meerle': 'Flanders',
    'Meix-le-Tige': 'Wallonia',
    'Meldert': 'Flanders',
    'Melen': 'Wallonia',
    'Mélin': 'Wallonia',
    'Mellet': 'Wallonia',
    'Mellier': 'Wallonia',
    'Merdorp': 'Wallonia',
    'Mere': 'Flanders',
    'Merksem': 'Flanders',
    'Merlemont': 'Wallonia',
    'Messancy': 'Wallonia',
    'Meux': 'Wallonia',
    'Michelbeke': 'Flanders',
    'Micheroux': 'Wallonia',
    'Milmort': 'Wallonia',
    'Minderhout': 'Flanders',
    'Moen': 'Flanders',
    'Moerbeke-Waas': 'Flanders',
    'Moerbeke': 'Flanders',
    'Moignelée': 'Wallonia',
    'Moha': 'Wallonia',
    'Molenbaix': 'Wallonia',
    'Molenbeek-Wersbeek': 'Flanders',
    'Molenbeersel': 'Flanders',
    'Molenstede': 'Flanders',
    'Monceau-sur-Sambre': 'Wallonia',
    'Monstreux': 'Wallonia',
    'Mont': 'Wallonia',
    'Mont-Saint-André': 'Wallonia',
    'Mont-Saint-Aubert': 'Wallonia',
    'Mont-Sainte-Aldegonde': 'Wallonia',
    'Mont-sur-Marchienne': 'Wallonia',
    'Montegnée': 'Wallonia',
    'Montignies-sur-Roc': 'Wallonia',
    'Montignies-sur-Sambre': 'Wallonia',
    'Moorsele': 'Flanders',
    'Moorsel': 'Flanders',
    'Moregem': 'Flanders',
    'Morialmé': 'Wallonia',
    'Morkhoven': 'Flanders',
    'Morlanwelz-Mariemont': 'Wallonia',
    'Mortroux': 'Wallonia',
    'Mourcourt': 'Wallonia',
    'Moustier-Sur-Sambre': 'Wallonia',
    'Mouzaive': 'Wallonia',
    'Muizen': 'Flanders',
    'Munsterbilzen': 'Flanders',
    'Mussy-la-Ville': 'Wallonia',

    # N
    'Naast': 'Wallonia',
    'Nalinnes': 'Wallonia',
    'Namêche': 'Wallonia',
    'Naninne': 'Wallonia',
    'Néchin': 'Wallonia',
    'Neder-Over-Heembeek': 'Brussels-Capital',
    'Nederhasselt': 'Flanders',
    'Neerheylissem': 'Wallonia',
    'Neerijse': 'Flanders',
    'Neeroeteren': 'Flanders',
    'Neigem': 'Flanders',
    'Nessonvaux': 'Wallonia',
    'Nettinne': 'Wallonia',
    'Neufmaison': 'Wallonia',
    'Neufvilles': 'Wallonia',
    'Neuville': 'Wallonia',
    'Neuville-en-Condroz': 'Wallonia',
    'Niel-bij-As': 'Flanders',
    'Nieuwenhove': 'Flanders',
    'Nieuwkerke': 'Flanders',
    'Nieuwkerken-Waas': 'Flanders',
    'Nil-Saint-Vincent-Saint-Martin': 'Wallonia',
    'Nimy': 'Wallonia',
    'Nismes': 'Wallonia',
    'Nodebais': 'Wallonia',
    'Noduwez': 'Wallonia',
    'Noiseux': 'Wallonia',
    'Noorderwijk': 'Flanders',
    'Nossegem': 'Flanders',
    'Nothomb': 'Wallonia',
    'Noville-les-Bois': 'Wallonia',
    'Nukerke': 'Flanders',

    # O
    'Obourg': 'Wallonia',
    'Ocquier': 'Wallonia',
    'Oelegem': 'Flanders',
    'Oevel': 'Flanders',
    'Oetingen': 'Flanders',
    'Offagne': 'Wallonia',
    'Ogy': 'Wallonia',
    'Oignies-En-Thiérache': 'Wallonia',
    'Oisquercq': 'Wallonia',
    'Oekene': 'Flanders',
    'Oleye': 'Wallonia',
    'Olmen': 'Flanders',
    'Olne': 'Wallonia',
    'Olsene': 'Flanders',
    'On': 'Wallonia',
    'Onnezies': 'Wallonia',
    'Onze-Lieve-Vrouw-Waver': 'Flanders',
    'Ooigem': 'Flanders',
    'Oordegem': 'Flanders',
    'Oostakker': 'Flanders',
    'Oostduinkerke': 'Flanders',
    'Oostmalle': 'Flanders',
    'Oostnieuwkerke': 'Flanders',
    'Opglabbeek': 'Flanders',
    'Opgrimbie': 'Flanders',
    'Ophain-Bois-Seigneur-Isaac': 'Wallonia',
    'Orcq': 'Wallonia',
    'Ormeignies': 'Wallonia',
    'Orp-le-Grand': 'Wallonia',
    'Orroir': 'Wallonia',
    'Ortho': 'Wallonia',
    'Ostiches': 'Wallonia',
    'Otegem': 'Flanders',
    'Ottignies': 'Wallonia',
    'Oudergem': 'Brussels-Capital',
    'Oudenaken': 'Flanders',
    'Oudsbergen': 'Flanders',
    'Ougrée': 'Wallonia',
    'Outer': 'Flanders',
    'Ouwegem': 'Flanders',
    'Overboelare': 'Flanders',
    'Overpelt': 'Flanders',

    # P
    'Paal': 'Flanders',
    'Parike': 'Flanders',
    'Passendale': 'Flanders',
    'Pâturages': 'Wallonia',
    'Peer': 'Flanders',
    'Pellaines': 'Wallonia',
    'Pellenberg': 'Flanders',
    'Péronnes': 'Wallonia',
    'Petit-Enghien': 'Wallonia',
    'Petit-Hallet': 'Wallonia',
    'Petit-Rechain': 'Wallonia',
    'Petit-Thier': 'Wallonia',
    'Petit-Roeulx-lez-Braine': 'Wallonia',
    'PHILIPPEVILLE': 'Wallonia',
    'Piéton': 'Wallonia',
    'Piétrain': 'Wallonia',
    'Pipaix': 'Wallonia',
    'Plainevaux': 'Wallonia',
    'Ploegsteert': 'Wallonia',
    'Poederlee': 'Flanders',
    'Poelkapelle': 'Flanders',
    'Pollare': 'Flanders',
    'Pommeroeul': 'Wallonia',
    'Pont-de-Loup': 'Wallonia',
    'Poppel': 'Flanders',
    'Presles': 'Wallonia',
    'Pry': 'Wallonia',
    'Pulderbos': 'Flanders',
    'Puurs': 'Flanders',

    # Q
    'Quenast': 'Wallonia',
    'Queue-du-Bois': 'Wallonia',
    'Quevaucamps': 'Wallonia',

    # R
    'Rachecourt': 'Wallonia',
    'Ramegnies': 'Wallonia',
    'Ramegnies-Chin': 'Wallonia',
    'Ramillies-Offus': 'Wallonia',
    'Ramsel': 'Flanders',
    'Ramskapelle': 'Flanders',
    'Rance': 'Wallonia',
    'Ransart': 'Wallonia',
    'Recht': 'Wallonia',
    'Rekem': 'Flanders',
    'Rekkem': 'Flanders',
    'Relegem': 'Flanders',
    'Remersdaal': 'Flanders',
    'Reninge': 'Flanders',
    'Renlies': 'Wallonia',
    'Ressaix': 'Wallonia',
    'Ressegem': 'Flanders',
    'Retinne': 'Wallonia',
    'Rhisnes': 'Wallonia',
    'Richelle': 'Wallonia',
    'Rièzes': 'Wallonia',
    'Rijmenam': 'Flanders',
    'Rillaar': 'Flanders',
    'Robertville': 'Wallonia',
    'Roclenge-sur-Geer': 'Wallonia',
    'Roesbrugge-Haringe': 'Flanders',
    'Roisin': 'Wallonia',
    'Roly': 'Wallonia',
    'Romedenne': 'Wallonia',
    'Romsée': 'Wallonia',
    'Rongy': 'Wallonia',
    'Rosières': 'Wallonia',
    'Rosoux-Crenwick': 'Wallonia',
    'Roucourt': 'Wallonia',
    'Roux': 'Wallonia',
    'Rozebeke': 'Flanders',
    'Ruddervoorde': 'Flanders',
    'Ruisbroek': 'Flanders',
    'Rumillies': 'Wallonia',
    'Rutten': 'Flanders',

    # S
    's-Gravenvoeren': 'Flanders',
    's-Gravenwezel': 'Flanders',
    'Saint-André': 'Wallonia',
    'Saint-Aubin': 'Wallonia',
    'Saint-Jean-Geest': 'Wallonia',
    'Saint-Marc': 'Wallonia',
    'Saint-Mard': 'Wallonia',
    'Saint-Martin': 'Wallonia',
    'Saint-Remy': 'Wallonia',
    'Saint-Sauveur': 'Wallonia',
    'Saint-Servais': 'Wallonia',
    'Saint-Symphorien': 'Wallonia',
    'Sainte-Cécile': 'Wallonia',
    'Sainte-Ode': 'Wallonia',
    'Saintes': 'Wallonia',
    'Saive': 'Wallonia',
    'Sars-la-Bruyère': 'Wallonia',
    'Sars-la-Buissière': 'Wallonia',
    'Sart-Custinne': 'Wallonia',
    'Sart-Dames-Avelines': 'Wallonia',
    'Sart-Saint-Laurent': 'Wallonia',
    'Sautour': 'Wallonia',
    'Sauvenière': 'Wallonia',
    'Schaarbeek': 'Brussels-Capital',
    'Schellebelle': 'Flanders',
    'Schepdaal': 'Flanders',
    'Scherpenheuvel': 'Flanders',
    'Schorisse': 'Flanders',
    'Schriek': 'Flanders',
    'Sclayn': 'Wallonia',
    'Seilles': 'Wallonia',
    'Seloignes': 'Wallonia',
    'Serskamp': 'Flanders',
    'Sibret': 'Wallonia',
    'Sijsele': 'Flanders',
    'Silenrieux': 'Wallonia',
    'Sinaai': 'Flanders',
    'Sinsin': 'Wallonia',
    'Sint-Agatha-Rode': 'Flanders',
    'Sint-Amandsberg': 'Flanders',
    'Sint-Andries': 'Flanders',
    'Sint-Baafs-Vijve': 'Flanders',
    'Sint-Blasius-Boekel': 'Flanders',
    'Sint-Denijs': 'Flanders',
    'Sint-Denijs-Boekel': 'Flanders',
    'Sint-Denijs-Westrem': 'Flanders',
    'Sint-Eloois-Vijve': 'Flanders',
    'Sint-Gillis': 'Brussels-Capital',
    'Sint-Gillis-bij-Dendermonde': 'Flanders',
    'Sint-Idesbald': 'Flanders',
    'Sint-Jans-Molenbeek': 'Brussels-Capital',
    'Sint-Job-in-\'t-Goor': 'Flanders',
    'Sint-Joost-ten-Node': 'Brussels-Capital',
    'Sint-Joris-Weert': 'Flanders',
    'Sint-Joris-Winge': 'Flanders',
    'Sint-Katherina-Lombeek': 'Flanders',
    'Sint-Kruis': 'Flanders',
    'Sint-Lambrechts-Woluwe': 'Brussels-Capital',
    'Sint-Maria-Oudenhove': 'Flanders',
    'Sint-Martens-Lierde': 'Flanders',
    'Sint-Michiels': 'Flanders',
    'Sint-Pauwels': 'Flanders',
    'Sint-Pieters-Kapelle': 'Flanders',
    'Sint-Pieters-Woluwe': 'Brussels-Capital',
    'Sirault': 'Wallonia',
    'Sivry': 'Wallonia',
    'Sleidinge': 'Flanders',
    'Slins': 'Wallonia',
    'Sohier': 'Wallonia',
    'Soiron': 'Wallonia',
    'Solre-Saint-Géry': 'Wallonia',
    'Somzée': 'Wallonia',
    'Sorinnes': 'Wallonia',
    'Sougné-Remouchamps': 'Wallonia',
    'Soulme': 'Wallonia',
    'Soumoy': 'Wallonia',
    'Souvret': 'Wallonia',
    'Soy': 'Wallonia',
    'Spiennes': 'Wallonia',
    'Stambruges': 'Wallonia',
    'Steendorp': 'Flanders',
    'Steenhuffel': 'Flanders',
    'Steenkerque': 'Wallonia',
    'Stembert': 'Wallonia',
    'Sterrebeek': 'Flanders',
    'Stokkem': 'Flanders',
    'Strée': 'Wallonia',
    'Strépy-Bracquegnies': 'Wallonia',
    'Strombeek-Bever': 'Flanders',
    'Sugny': 'Wallonia',

    # T
    'Taintignies': 'Wallonia',
    'Tamines': 'Wallonia',
    'Tarcienne': 'Wallonia',
    'Tavier': 'Wallonia',
    'Templeuve': 'Wallonia',
    'Temploux': 'Wallonia',
    'Teralfene': 'Flanders',
    'Tertre': 'Wallonia',
    'Testelt': 'Flanders',
    'Thieu': 'Wallonia',
    'Thorembais-Saint-Trond': 'Wallonia',
    'Thoricourt': 'Wallonia',
    'Thulin': 'Wallonia',
    'Thumaide': 'Wallonia',
    'Thuillies': 'Wallonia',
    'Thy-le-Bauduin': 'Wallonia',
    'Thy-le-Château': 'Wallonia',
    'Thynes': 'Wallonia',
    'Tiegem': 'Flanders',
    'Tielen': 'Flanders',
    'Tihange': 'Wallonia',
    'Tilff': 'Wallonia',
    'Tilleur': 'Wallonia',
    'Tilly': 'Wallonia',
    'Tisselt': 'Flanders',
    'Tollembeek': 'Flanders',
    'Tongerlo': 'Flanders',
    'Tongrinne': 'Wallonia',
    'Torgny': 'Wallonia',
    'Tourinnes-Saint-Lambert': 'Wallonia',
    'Trazegnies': 'Wallonia',
    'Trivières': 'Wallonia',

    # U
    'Ukkel': 'Brussels-Capital',

    # V
    'Vance': 'Wallonia',
    'Varendonk': 'Flanders',
    'Varsenare': 'Flanders',
    'Vaulx': 'Wallonia',
    'Vaux-et-Borset': 'Wallonia',
    'Vaux-sous-Chèvremont': 'Wallonia',
    'Vedrin': 'Wallonia',
    'Veerle': 'Flanders',
    'Velaine-sur-Sambre': 'Wallonia',
    'Velaines': 'Wallonia',
    'Veldegem': 'Flanders',
    'Veldwezelt': 'Flanders',
    'Vencimont': 'Wallonia',
    'Vezin': 'Wallonia',
    'Vezon': 'Wallonia',
    'Viane': 'Wallonia',
    'Viesville': 'Wallonia',
    'Vieux-Genappe': 'Wallonia',
    'Ville-Pommeroeul': 'Wallonia',
    'Villers-Poterie': 'Wallonia',
    'Villers-Saint-Amand': 'Wallonia',
    'Villers-le-Gambon': 'Wallonia',
    'Villers-le-Peuplier': 'Wallonia',
    'Villers-le-Temple': 'Wallonia',
    'Villers-l\'Évêque': 'Wallonia',
    'Villers-sur-Semois': 'Wallonia',
    'Vinalmont': 'Wallonia',
    'Vinderhoute': 'Flanders',
    'Virelles': 'Wallonia',
    'Virginal-Samme': 'Wallonia',
    'Vitrival': 'Wallonia',
    'Vivegnis': 'Wallonia',
    'Vlamertinge': 'Flanders',
    'Vlezenbeek': 'Flanders',
    'Vliermaal': 'Flanders',
    'Vlimmeren': 'Flanders',
    'Vodecée': 'Wallonia',
    'Vodelée': 'Wallonia',
    'Voormezele': 'Flanders',
    'Vonêche': 'Wallonia',
    'Voroux-lez-Liers': 'Wallonia',
    'Vorst': 'Brussels-Capital',
    'Vottem': 'Wallonia',
    'Vrasene': 'Flanders',
    'Vremde': 'Flanders',
    'Vyle-et-Tharoul': 'Wallonia',

    # W
    'Waarbeke': 'Flanders',
    'Waardamme': 'Flanders',
    'Waarloos': 'Flanders',
    'Wagnelée': 'Wallonia',
    'Waha': 'Wallonia',
    'Wakken': 'Flanders',
    'Walhain-Saint-Paul': 'Wallonia',
    'Wambeek': 'Flanders',
    'Wanfercée-Baulet': 'Wallonia',
    'Wanlin': 'Wallonia',
    'Waret-l\'Évêque': 'Wallonia',
    'Warcoing': 'Wallonia',
    'Warneton': 'Wallonia',
    'Warsage': 'Wallonia',
    'Wasmes': 'Wallonia',
    'Wasmuel': 'Wallonia',
    'Watermaal-Bosvoorde': 'Brussels-Capital',
    'Watervliet': 'Flanders',
    'Waudrez': 'Wallonia',
    'Waulsort': 'Wallonia',
    'Wauthier-Braine': 'Wallonia',
    'Wavreille': 'Wallonia',
    'Wechelderzande': 'Flanders',
    'Weelde': 'Flanders',
    'Weerde': 'Flanders',
    'Wegnez': 'Wallonia',
    'Welle': 'Flanders',
    'Wenduine': 'Flanders',
    'Werbomont': 'Wallonia',
    'Werchter': 'Flanders',
    'Wéris': 'Wallonia',
    'Westende': 'Flanders',
    'Westkerke': 'Flanders',
    'Westmalle': 'Flanders',
    'Westmeerbeek': 'Flanders',
    'Westouter': 'Flanders',
    'Westrem': 'Flanders',
    'Westrozebeke': 'Flanders',
    'Wezemaal': 'Flanders',
    'Wibrin': 'Wallonia',
    'Wiekevorst': 'Flanders',
    'Wierde': 'Wallonia',
    'Wiers': 'Wallonia',
    'Wijchmaal': 'Flanders',
    'Wijgmaal': 'Flanders',
    'Wihéries': 'Wallonia',
    'Willemeau': 'Wallonia',
    'Wilrijk': 'Flanders',
    'Wilsele': 'Flanders',
    'Woesten': 'Flanders',
    'Wolvertem': 'Flanders',
    'Woluwe-saint-Etienne': 'Flanders',
    'Wonck': 'Wallonia',
    'Wondelgem': 'Flanders',
    'Wulvergem': 'Flanders',

    # X
    'Xhoris': 'Wallonia',

    # Y
    'Yves-Gomezée': 'Wallonia',

    # Z
    'Zaffelare': 'Flanders',
    'Zandbergen': 'Flanders',
    'Zarren': 'Flanders',
    'Zeebrugge': 'Flanders',
    'Zelem': 'Flanders',
    'Zellik': 'Flanders',
    'Zétrud-Lumay': 'Wallonia',
    'Zeveneken': 'Flanders',
    'Zillebeke': 'Flanders',
    'Zoerle-Parwijs': 'Flanders',
    'Zolder': 'Flanders',
    'Zwevezele': 'Flanders',
    'Zwijnaarde': 'Flanders',
    
    'Achet': 'Wallonia',
    
    # B
    'Baileux': 'Wallonia',
    'Bevercé': 'Wallonia',
    
    # E
    'Ében-Émael': 'Wallonia',
    'Écaussinnes-d\'Enghien': 'Wallonia',
    
    # H
    'Houdremont': 'Wallonia',
    
    # L
    'Leeuwergem': 'Flanders',
    'Lessive': 'Wallonia',
    'Liege': 'Wallonia',
    
    # M
    'Marche-lez-Écaussinnes': 'Wallonia',
    'Merbes-le-Château': 'Wallonia',
    'Moelingen': 'Flanders',
    'Mons-Lez-Liège': 'Wallonia',
    
    # W
    'Wépion': 'Wallonia'

    
}


In [8]:
lookup = {}
for name, region in municipality_to_region.items():
    norm = name
    # In case of duplicates, keep the first (should not happen)
    if norm not in lookup:
        lookup[norm] = region

# Function to get region for a locality
def get_region(locality):
    norm_loc = locality
    # First try exact match
    if norm_loc in lookup:
        return lookup[norm_loc]
    # If not found, try to see if locality contains a municipality name?
    # For simplicity, we'll just return Unknown and print it.
    return 'Unknown'

# Apply to dataframe
df['region'] = df['locality'].apply(get_region)

# Check for unknowns
unknowns = df[df['region'] == 'Unknown']['locality'].unique()

In [9]:
unknown_region_df = df[df['region'] == 'Unknown']
unknown_region_df

,locality,type,subtype,price,living_area,land_area,facades,state,furnished,terrace,garden,pool,region


In [10]:
dataset_clean = df.drop_duplicates()
dataset_clean.duplicated().sum()
dataset_clean

,locality,type,subtype,price,living_area,land_area,facades,state,furnished,terrace,garden,pool,region
0,Ladeuze,house,residence,90000,120,235,2,2,False,True,True,False,Wallonia
1,Charleroi,apartment,apartment,90000,49,0,2,6,False,True,False,False,Wallonia
2,Anvaing,house,residence,90000,165,1410,4,2,False,False,True,False,Wallonia
3,Dour,house,residence,90000,115,269,2,2,False,True,False,False,Wallonia
4,Élouges,house,residence,90000,113,170,2,2,False,True,True,False,Wallonia
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14608,Sint-Pieters-Woluwe,house,villa,2250000,365,1571,4,4,False,True,True,False,Brussels-Capital
14609,Sint-Pieters-Woluwe,house,villa,2250000,375,1571,4,7,False,True,True,False,Brussels-Capital
14610,Ukkel,apartment,penthouse,2250000,330,0,4,6,False,True,True,True,Brussels-Capital
14611,Sint-Pieters-Woluwe,house,villa,2250000,365,1571,4,7,False,True,True,False,Brussels-Capital


In [11]:
unique_localities = sorted(df['locality'].unique())
len(unique_localities)

1531